In [ ]:
from xml.dom.minidom import parse, parseString

In [ ]:
dom = parse('tvguide-ds.xml')

8.1
a)


In [ ]:
# finding all movies
root = dom.documentElement
movies = dom.getElementsByTagName("movie")
movies_count = len(movies)
print("Total number of movies: " + str(movies_count))

Total number of movies: 98


In [ ]:
for movie in movies:
  movie1 = movie.getElementsByTagName("name")[0].firstChild.data
print(movie1)

Spaceballs


In [ ]:
# Extracts all details of a movie
def get_details(movie):
    movie_data = {}

    for node in movie.childNodes:
        if node.nodeType == node.ELEMENT_NODE:
            tag = node.tagName

            # Check for attributes
            attributes = {attr: node.getAttribute(attr) for attr in node.attributes.keys()} if node.hasAttributes() else None

            # Get text values
            child_values = [child.nodeValue.strip() for child in node.childNodes if child.nodeType == child.TEXT_NODE and child.nodeValue.strip()]

            # Check for nested elements
            nested_data = get_details(node) if any(child.nodeType == node.ELEMENT_NODE for child in node.childNodes) else None

            # Store data appropriately
            if tag in movie_data:
                if not isinstance(movie_data[tag], list):
                    movie_data[tag] = [movie_data[tag]]
                movie_data[tag].append(nested_data if nested_data else " ".join(child_values))
            else:
                if nested_data:
                    if child_values:
                        nested_data["value"] = " ".join(child_values)
                    movie_data[tag] = nested_data
                elif child_values:
                    movie_data[tag] = child_values[0] if len(child_values) == 1 else child_values

            if attributes:
                movie_data[f"{tag}_attributes"] = attributes

    return movie_data


In [ ]:
get_details(movies[3])

{'name': '88 Minutes',
 'rank': '86',
 'year': '2008',
 'rating': 'R',
 'country': 'U.S.',
 'running-time': '108',
 'format': 'Color',
 'genres': {'genre': 'Thriller'},
 'production-companies': {'production-company': ['Equity Pictures Medienfonds',
   'Millennium Films',
   'Nu Image Entertainment']},
 'released-by': 'Sony',
 'user-rating': {'rating': '3.5', 'support': '10'},
 'actor-list': {'actor': [{'name': 'Carrie Genzel',
    'role': 'Stephanie Parkman'},
   {'name': 'Christopher Redman', 'role': 'Jeremy Guber'},
   {'name': 'Tim Henry', 'role': 'Sean McBain'},
   {'name': 'Brendan Fletcher', 'role': "Johnny D'Franco"},
   {'name': 'Kaj-Erik Eriksen', 'role': 'Matt Wilner'},
   {'name': 'Timothy Perez', 'role': 'Cabbie #1'},
   {'name': 'Leelee Sobieski', 'role': 'Lauren Douglas'},
   {'name': 'Kristina Copeland', 'role': 'Dale Morris'},
   {'name': 'Brenda McDonald', 'role': 'Mrs. Lowinsky'},
   {'name': 'Michal Yannai', 'role': 'Leeza Pearson'},
   {'name': 'Julian D. Christophe

b)

In [ ]:
from xml.sax import parse
from xml.sax.handler import ContentHandler

class IMDBHandler(ContentHandler):
    def __init__(self):
        self.current_tag = None
        self.string_builder = []
        self.current_movie = {}
        self.movies = []
        self.movie_count = 0

    def startElement(self, name, attrs):
        self.current_tag = name
        if name == "movie":
            self.current_movie = {}
            self.movie_count += 1

            # Store movie attributes
            if attrs.getLength() > 0:
                self.current_movie["attributes"] = {attr: attrs.getValue(attr) for attr in attrs.getNames()}

    def endElement(self, name):
        if name == "movie":
            self.movies.append(self.current_movie)
        elif self.current_tag and self.string_builder:
            value = " ".join(self.string_builder).strip()

            # Store movie details
            if name in self.current_movie:
                if isinstance(self.current_movie[name], list):
                    self.current_movie[name].append(value)
                else:
                    self.current_movie[name] = [self.current_movie[name], value]
            else:
                self.current_movie[name] = value

        self.current_tag = None
        self.string_builder = []

    def characters(self, content):
        if self.current_tag and content.strip():
            self.string_builder.append(content.strip())

# Create handler instance and parse XML
handler = IMDBHandler()
try:
    parse("imdb-ds.xml", handler)
except Exception as e:
    print("Parsing stopped:", e)


print(f"Total movies parsed: {handler.movie_count}")


Total movies parsed: 243856


In [ ]:
handler.movies[985]

{'attributes': {'imdbid': 'idm423755831520'},
 'title': 'Mistaken Identity',
 'genre': ['Comedy', 'Short'],
 'name': ['Douglas, Sean (I)',
  'George, Martin (I)',
  'Liebe, Christopher',
  'Max, Don (I)',
  'Page, Madden',
  'Romano, Joe (IV)',
  'Stasi, Matt',
  'White, Michael (XVIII)',
  'Green-Gaber, Renata',
  'Mauro, Christina'],
 'role': ['Crandall',
  'Griffen',
  'Dashing Gentleman',
  'Mr. Wonderful',
  'Interregator',
  'Bathroom Guy',
  'Lone Gunman',
  'Office Suit',
  'Margaret',
  'Carly'],
 'director': 'Green-Gaber, Renata',
 'location': 'Los Angeles, California, USA',
 'year': '2004'}

c)

In [ ]:
#!pip install jellyfish

In [ ]:
from xml.dom.minidom import parse, parseString
import xml.dom.minidom as minidom
import xml.sax
import jellyfish
from itertools import product

In [ ]:
# Parsing IMDB XML using SAX and extracts movie titles
class IMDBHandler_titles(xml.sax.ContentHandler):
    def __init__(self):
        self.current_tag = None
        self.title = None
        self.titles = []

    def startElement(self, name, attrs):
        self.current_tag = name
        if name == "movie":
            self.title = None  # Reset title

    def endElement(self, name):
        if name == "title" and self.title:
            self.titles.append(self.title.strip())
        if len(self.titles) >= 5:  # Stop after 5 titles
            raise Exception("Limit reached")

    def characters(self, content):
        if self.current_tag == "title":
            self.title = (self.title or "") + content


In [ ]:
#for parsing the TV Guide
def parse_tv_guide(file):
    doc = minidom.parse(file)
    movies = doc.getElementsByTagName("movie")
    titles = [movie.getElementsByTagName("name")[0].firstChild.data for movie in movies[:5]]
    return titles

In [ ]:
#for parsing IMDB
def parse_imdb(file):
    handler = IMDBHandler_titles()
    try:
        xml.sax.parse(file, handler)
    except Exception:
        pass
    return handler.titles

In [ ]:
def calculate_similarity(tv_titles, imdb_titles):
    def levenshtein(a, b):
        return jellyfish.levenshtein_distance(a, b)

    def jaro_winkler(a, b):
        return jellyfish.jaro_winkler_similarity(a, b)

    def hamming(a, b):
        return jellyfish.hamming_distance(a.ljust(max(len(a), len(b))), b.ljust(max(len(a), len(b))))

    def soundex_similarity(a, b):
        soundex_a = jellyfish.soundex(a)
        soundex_b = jellyfish.soundex(b)
        jw_score = jaro_winkler(a, b)  # we chose to use Jaro-Winkler as extra measure
        return jw_score if soundex_a == soundex_b else 0

    metrics = {
        "Levenshtein Distance": levenshtein,
        "Jaro-Winkler Similarity": jaro_winkler,
        "Hamming Distance": hamming,
        "Soundex": soundex_similarity
    }

    for metric_name, metric in metrics.items():
        print(f"\n {metric_name} Scores:")
        for tv, imdb in product(tv_titles, imdb_titles):
            score = metric(tv, imdb)
            print(f"- '{tv}' vs '{imdb}': {score}")


tv_titles = parse_tv_guide("tvguide-ds.xml")
imdb_titles = parse_imdb("imdb-ds.xml")

print("TV Guide Titles:", tv_titles)
print("IMDB Titles:", imdb_titles)

calculate_similarity(tv_titles, imdb_titles)


TV Guide Titles: ['Ever After', 'First Daughter', 'Hope Floats', '88 Minutes', 'Chicago']
IMDB Titles: ['Katka i Shiz', 'Prisoner of Zenda', 'Dough for the Do-Do', 'Bad Credit and Aliens', 'Texana cien X #3']

 Levenshtein Distance Scores:
- 'Ever After' vs 'Katka i Shiz': 11
- 'Ever After' vs 'Prisoner of Zenda': 12
- 'Ever After' vs 'Dough for the Do-Do': 16
- 'Ever After' vs 'Bad Credit and Aliens': 17
- 'Ever After' vs 'Texana cien X #3': 15
- 'First Daughter' vs 'Katka i Shiz': 12
- 'First Daughter' vs 'Prisoner of Zenda': 15
- 'First Daughter' vs 'Dough for the Do-Do': 17
- 'First Daughter' vs 'Bad Credit and Aliens': 17
- 'First Daughter' vs 'Texana cien X #3': 15
- 'Hope Floats' vs 'Katka i Shiz': 11
- 'Hope Floats' vs 'Prisoner of Zenda': 14
- 'Hope Floats' vs 'Dough for the Do-Do': 16
- 'Hope Floats' vs 'Bad Credit and Aliens': 17
- 'Hope Floats' vs 'Texana cien X #3': 15
- '88 Minutes' vs 'Katka i Shiz': 11
- '88 Minutes' vs 'Prisoner of Zenda': 15
- '88 Minutes' vs 'Dough f

d)

In [ ]:
threshold = 0.85
output_file = "matched_movies.txt"

In [ ]:
def jaro_winkler(a, b):
    return jellyfish.jaro_winkler_similarity(a, b)

In [ ]:
class IMDBHandler_d(xml.sax.ContentHandler):
    def __init__(self):
        self.current_tag = None
        self.title = None
        self.titles = []

    def startElement(self, name, attrs):
        self.current_tag = name
        if name == "movie":
            self.title = None  # Reset title

    def endElement(self, name):
        if name == "title" and self.title:
            self.titles.append(self.title.strip())  # Store title

    def characters(self, content):
        if self.current_tag == "title":
            self.title = (self.title or "") + content  # Store movie title

def parse_imdbb(file):
    handler = IMDBHandler_d()
    xml.sax.parse(file, handler)
    return handler.titles

def parse_tv_guide_all(file):
    doc = minidom.parse(file)
    movies = doc.getElementsByTagName("movie")
    titles = [movie.getElementsByTagName("name")[0].firstChild.data for movie in movies]
    return titles

In [ ]:
def find_matches(tv_titles, imdb_titles, threshold, output_file):

    with open(output_file, "w", encoding="utf-8") as f:
        for tv_title in tv_titles:
            best_match = None
            best_score = 0

            for imdb_title in imdb_titles:
                score = jaro_winkler(tv_title, imdb_title)

                if score > best_score:
                    best_score = score
                    best_match = imdb_title

            if best_score >= threshold:
                f.write(f"{tv_title} -> {best_match} (Score: {best_score:.4f})\n")

    print("see matching in the output file")

In [ ]:
tv_titles = parse_tv_guide_all("tvguide-ds.xml")
imdb_titles = parse_imdbb("imdb-ds.xml")

find_matches(tv_titles, imdb_titles, threshold, output_file)

see matching in the output file


e)

In [ ]:
from collections import defaultdict
def find_best_unique_matches(tv_titles, imdb_titles, threshold, output_file):
    match_candidates = []

    # find the best match for each TV movie
    for tv_title in tv_titles:
        best_match = None
        best_score = 0

        for imdb_title in imdb_titles:
            score = jaro_winkler(tv_title, imdb_title)

            if score > best_score:
                best_score = score
                best_match = imdb_title

        if best_score >= 0.5:
            match_candidates.append((tv_title, best_match, best_score))

    # make sure imdb movies are assigned uniquely
    assigned_imdb = set()
    final_matches = []

    # Sort matches by score
    match_candidates.sort(key=lambda x: x[2], reverse=True)

    for tv_title, imdb_title, score in match_candidates:
        if imdb_title not in assigned_imdb:
            final_matches.append((tv_title, imdb_title, score))
            assigned_imdb.add(imdb_title)

    # write final matches to file
    with open(output_file, "w", encoding="utf-8") as f:
        for tv_title, imdb_title, score in final_matches:
            f.write(f"{tv_title} -> {imdb_title} (Score: {score:.4f})\n")

    print("Unique best matches written to output file")


tv_titles = parse_tv_guide_all("tvguide-ds.xml")
imdb_titles = parse_imdbb("imdb-ds.xml")

find_best_unique_matches(tv_titles, imdb_titles, threshold, "unique_matches.txt")

Unique best matches written to output file


8.2) a)

In [ ]:
import jellyfish
from collections import defaultdict

# Function to compute different similarity scores
def compute_similarity(tv_title, imdb_title, method):
    if method == "jaro_winkler":
        return jellyfish.jaro_winkler_similarity(tv_title, imdb_title)
    elif method == "levenshtein":
        max_len = max(len(tv_title), len(imdb_title))
        return 1 - (jellyfish.levenshtein_distance(tv_title, imdb_title) / max_len)
    elif method == "hamming":
        if len(tv_title) != len(imdb_title):
            return 0
        return 1 - (jellyfish.hamming_distance(tv_title, imdb_title) / max(len(tv_title), len(imdb_title)))
    elif method == "soundex":
      soundex_a = jellyfish.soundex(tv_title)
      soundex_b = jellyfish.soundex(imdb_title)
      jw_score = jellyfish.jaro_winkler_similarity(tv_title, imdb_title)  # we chose to use Jaro-Winkler as extra measure
      return jw_score if soundex_a == soundex_b else 0
    else:
        raise ValueError("Invalid similarity method")

def find_best_unique_matches(tv_titles, imdb_titles, threshold, output_file_prefix):
    similarity_methods = ["levenshtein", "jaro_winkler", "hamming", "soundex"]
    for method in similarity_methods:
        match_candidates = []  # To store all possible matches for each similarity function

        # find the best match for each TV movie
        for tv_title in tv_titles:
            best_match = None
            best_score = 0

            for imdb_title in imdb_titles:
                score = compute_similarity(tv_title, imdb_title, method)

                if score > best_score:
                    best_score = score
                    best_match = imdb_title

            if best_score >= threshold:
                match_candidates.append((tv_title, best_match, best_score))

        assigned_imdb = set()
        final_matches = []


        for tv_title, imdb_title, score in match_candidates:
            if imdb_title not in assigned_imdb:
                final_matches.append((tv_title, imdb_title, score))
                assigned_imdb.add(imdb_title)

        # Write final matches to different files based on similarity function
        output_file = f"{output_file_prefix}_{method}_matches.txt"
        with open(output_file, "w", encoding="utf-8") as f:
            for tv_title, imdb_title, score in final_matches:
                f.write(f"{tv_title},{imdb_title},{score:.4f}\n")

        print(f"Matches using {method} similarity saved to {output_file}")



tv_titles = parse_tv_guide_all("tvguide-ds.xml")
imdb_titles = parse_imdbb("imdb-ds.xml")

threshold = 0.0  # Adjust the threshold => 0 for now because we wanted to see all TV Guide movies matched
find_best_unique_matches(tv_titles, imdb_titles, threshold, "unique_matches")


In [ ]:
#function to separate IMDB movies from matching

def resultIMDBtitles(path):
  f = open(path, "r")
  titlesIMDB = []
  for line in f.readlines():
    titlesIMDB.append(line.strip().split(",")[1])
  return titlesIMDB



In [ ]:
titles_hamming = resultIMDBtitles("/content/unique_matches_hamming_matches.txt")
titles_jaro = resultIMDBtitles("/content/unique_matches_jaro_winkler_matches.txt")
titles_lev = resultIMDBtitles("/content/unique_matches_levenshtein_matches.txt")
titles_soundex = resultIMDBtitles("/content/unique_matches_soundex_matches.txt")

In [ ]:
#function to get IDs for IMDB titles

import xml.etree.ElementTree as ET

def parse_imdb_ids(file_path):
    imdb_ids = {}

    tree = ET.parse(file_path)
    root = tree.getroot()

    for movie in root.findall("movie"):
        imdbid = movie.attrib.get("imdbid")
        title_element = movie.find("title")

        if imdbid and title_element is not None:
            title = title_element.text.strip()
            imdb_ids[title] = imdbid

    return imdb_ids

def get_imdb_ids_for_titles(imdb_titles, imdb_mapping):
    """Finds IMDB IDs for a given list of movie titles."""
    result = [imdb_mapping.get(title, "Unknown") for title in imdb_titles]
    return result

imdb_mapping = parse_imdb_ids("imdb-ds.xml")

In [ ]:
print(titles_hamming)

['Ever After', 'First Daughter', 'Hope Floats', '88 Minutes', 'Chicago', "All the King's Men", 'Across the Universe', 'XX.', 'Outlaws of Sonora', 'Max Payne', 'Fly Me to the Moon', 'Blazing Saddles', 'Yentl', 'Changeling', 'Miss Congeniality', 'Casino Royale', 'Sky High', 'May daga sa labas ng lungga', 'American Samurai', 'Stars in My Crown', 'Body of Work', 'Two Minutes to Midnight', 'Ice Castles', 'Michael Clayton', 'Forever Lulu', 'Her Great Mistake', 'Black Snake Moan', '92 hak mooi gwai dui hak mooi gwai', "She She She She's a Bombshell", 'Tomorrow Never Dies', 'GoldenEye', 'Bindle', 'Raising Arizona', 'Foolproof', 'One Single Moment', 'Mamma Roma', 'Mrs. Taylor Going Over Horseshoe Falls in a Barrel', 'Tom and Jerry', ' Hidden Dragon', 'Batman Begins', 'Milk', 'Mandalay', 'While You Were Sleeping', 'Portrait of Jennie', 'Sail a Crooked Ship', 'Hocus Pocus', 'Man of the House', 'Hellboy', 'Miracle in the Rain', 'When Fate Leads Trump', 'Australia', 'Home Movies', 'Clockwatchers', 

In [ ]:
result_hamming = get_imdb_ids_for_titles(titles_hamming, imdb_mapping)
result_jaro = get_imdb_ids_for_titles(titles_jaro, imdb_mapping)
result_levenshtein = get_imdb_ids_for_titles(titles_lev, imdb_mapping)
result_soundex = get_imdb_ids_for_titles(titles_soundex, imdb_mapping)
print(len(result_hamming))

97


In [ ]:
#padding the prediction list so that we can use the sklearn evaluation function
def pad_predictions(y_true, y_predict):
    padded_y_predict = []

    for true_value in y_true:
        if true_value in y_predict:
            padded_y_predict.append(true_value)
        else:
            padded_y_predict.append('0')  # Pad missing values

    return padded_y_predict

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

relevant = []

with open ("ground_truth-1.txt","r") as f:
  for line in f.readlines():
    relevant.append(line.strip().split(",")[1])


results = [result_hamming, result_jaro, result_levenshtein, result_soundex]

for result in results:
  resulT = pad_predictions(relevant, result)
  preHamming,recHamming,f1Hamming,avHamming = precision_recall_fscore_support(relevant, resulT, average="micro")
  print("Precision: ",preHamming,"\nRecall:",recHamming,"\nF1 score:",f1Hamming,"\n")
  resulT = []

Precision:  0.4387755102040816 
Recall: 0.4387755102040816 
F1 score: 0.4387755102040816 

Precision:  0.5204081632653061 
Recall: 0.5204081632653061 
F1 score: 0.5204081632653061 

Precision:  0.5 
Recall: 0.5 
F1 score: 0.5 

Precision:  0.5 
Recall: 0.5 
F1 score: 0.5 



# b) Adding year and actors to matching

In [ ]:
#parsing the tv guide, but now with extra attributes

def parse_tv_guide_all_updated(file):
    doc = minidom.parse(file)
    movies = doc.getElementsByTagName("movie")

    movie_dict = {}  # {title: (id, year, set(actors))}

    for movie in movies:
        tv_id = movie.getAttribute("tvgid")
        name_nodes = movie.getElementsByTagName("name")
        year_nodes = movie.getElementsByTagName("year")
        actor_nodes = movie.getElementsByTagName("actor")

        title = name_nodes[0].firstChild.data.strip() if name_nodes else None
        year = int(year_nodes[0].firstChild.data.strip()) if year_nodes else None
        actors = {actor.getElementsByTagName("name")[0].firstChild.nodeValue.strip()
          for actor in actor_nodes
          if actor.getElementsByTagName("name") and actor.getElementsByTagName("name")[0].firstChild}


        if title:
            movie_dict[tv_id] = (title, year, actors)

    return movie_dict

In [ ]:
import xml.sax
from xml.sax.handler import ContentHandler

class IMDBHandlerUpdated(ContentHandler):
    def __init__(self):
        super().__init__()
        self.current_tag = None
        self.current_id = None
        self.current_title = None
        self.current_year = None
        self.current_actors = set()
        self.movie_dict = {}
        self.content_buffer = ""
        self.in_movie = False

    def startElement(self, name, attrs):
        self.current_tag = name
        if name == "movie":
            self.in_movie = True
            if attrs.get("imdbid"):
                self.current_id = attrs.get("imdbid")
            self.current_title = None
            self.current_year = None
            self.current_actors = set()

        self.content_buffer = ""

    def endElement(self, name):
        if self.in_movie:
            if name == "title":
                self.current_title = self.content_buffer.strip()
            elif name == "year":
                try:
                    self.current_year = int(self.content_buffer.strip())
                except (ValueError, TypeError):
                    self.current_year = None
            elif name == "name":
                if self.content_buffer.strip():
                    self.current_actors.add(self.content_buffer.strip())

            elif name == "movie":
                if self.current_id and self.current_title:
                    # store in the format (title, year, {actors})
                    self.movie_dict[self.current_id] = (
                        self.current_title,
                        self.current_year,
                        self.current_actors
                    )
                self.in_movie = False

        # reset buffer and current tag
        self.content_buffer = ""
        if name == self.current_tag:
            self.current_tag = None

    def characters(self, content):
        if self.current_tag:
            self.content_buffer += content

    def get_movie_dict(self):
        return self.movie_dict

# Function to parse the IMDB XML file
def parse_imdb_act_year(file):
    handler = IMDBHandlerUpdated()
    xml.sax.parse(file, handler)
    return handler.get_movie_dict()

In [ ]:
def year_similarity(tv_year, imdb_year):
    if tv_year is None or imdb_year is None:
        return 0  # no match ifis missing
    return 1 if tv_year == imdb_year else max(0, 1 - (abs(tv_year - imdb_year) / 10))  # we made a small penalty if the years are close

In [ ]:
def jaccard_similarity(tv_actors, imdb_actors):
    if not tv_actors or not imdb_actors:
        return 0
    intersection = len(tv_actors & imdb_actors)
    union = len(tv_actors | imdb_actors)
    return intersection / union if union > 0 else 0


In [ ]:
def compute_final_similarity(tv_data, imdb_data, method, weights):
    tv_title, tv_year, tv_actors = tv_data
    imdb_title, imdb_year, imdb_actors = imdb_data

    title_sim = compute_similarity(tv_title, imdb_title, method)
    year_sim = year_similarity(tv_year, imdb_year)
    actor_sim = jaccard_similarity(set(tv_actors), set(imdb_actors))

    # Weighted sum
    final_score = (
        weights["title"] * title_sim +
        weights["year"] * year_sim +
        weights["actors"] * actor_sim
    )

    return final_score

In [ ]:
def find_best_unique_matches_actorsyears(tv_data, imdb_data, threshold, output_file, weights):
    match_candidates = []

    for tv_entry in tv_data:
        tv_title, tv_year, tv_actors = tv_entry
        best_match = None
        best_score = 0

        for imdb_entry in imdb_data:
            imdb_title, imdb_year, imdb_actors = imdb_entry

            # compute final similarity score using Jaro-Winkler (it produced the best result before)
            score = compute_final_similarity(
                (tv_title, tv_year, tv_actors),
                (imdb_title, imdb_year, imdb_actors),
                method="jaro_winkler",
                weights=weights
            )

            if score > best_score:
                best_score = score
                best_match = imdb_title  # store just the IMDb title

        if best_score >= threshold and best_match:
            match_candidates.append((tv_title, best_match, best_score))  # (tv_title, imdb_title, score)

    # keep original TV guide order when assigning unique IMDB movies
    assigned_imdb_titles = set()
    final_matches = []

    for tv_title, imdb_title, score in match_candidates:
        if imdb_title not in assigned_imdb_titles:
            final_matches.append((tv_title, imdb_title, score))
            assigned_imdb_titles.add(imdb_title)

    # Write results to file
    with open(output_file, "w", encoding="utf-8") as f:
        for tv_title, imdb_title, score in final_matches:
            f.write(f"{tv_title},{imdb_title},{score:.4f}\n")

    print(f"Matches saved to {output_file}")
    return final_matches

In [ ]:
from itertools import islice

tv_data = parse_tv_guide_all_updated("tvguide-ds.xml")  # Extract (title, year, actors)
print(tv_data)

imdb_data = parse_imdb_act_year("imdb-ds.xml")  # Extract (title, year, actors, imdb_id)
print(dict(islice(imdb_data.items(), 2)))


{'2': ('Ever After', 1998, {'Jeanne Moreau', 'Mark Lewis', 'Dominic Rold', 'Matyelok Gibbs', 'Ricardo Cruz', 'Anjelica Huston', 'Dougray Scott', 'Virginia Garcia', 'Al Hunter Ashton', 'Francois Velter', 'Elizabeth Earl', 'Andrew Henderson', 'Erick Awanzino', 'Patrick Godfrey', 'Lee Ingley', 'Toby Jones', 'Kate Lansbury', 'Howard Attfield', 'Christian Marc', 'Ursula Jones', 'Timothy West', 'Ricki Cuttell', 'Drew Barrymore', 'Alex Pooley', 'Megan Dodds', 'Melanie Lynskey', "Richard O'Brien", 'Janet Henfrey', 'Peter Gunn', 'Amanda Walker', 'Elvira Stevenson', 'Rupam Maxwell', 'Anna Maguire', 'Tony Doyle', 'Walter Sparrow', 'John Walters', 'Judy Parfitt', 'Joerg Stadler', 'Jean-Pierre Mazieres', 'Susan Field', 'Jeroen Krabbe'}), '392': ('First Daughter', 2004, {'Andrea Avery', 'Nicole Avant', 'Melissa Rivers', 'Maria Quiban', 'Tim Liles', 'Forest Whitaker', 'Sophia Chang', 'Teck Holmes', 'Ted Garcia', 'Philip Boyd', 'Tore Birkedal', 'Jeff Michael', 'Barry Livingston', 'Dan Brinkle', 'Ken M

In [ ]:
weights = {"title": 0.4, "year": 0.4, "actors": 0.2}
find_best_unique_matches_actorsyears(tv_data.values(), imdb_data.values(), threshold=0.5, output_file="matches_jaro.txt", weights=weights)

Matches saved to matches_jaro.txt


[('Ever After', 'Ever After', 0.8),
 ('First Daughter', 'First Daughter', 0.8018181818181819),
 ('Hope Floats', 'Hope Floats', 0.8),
 ('88 Minutes', '88 Minutes', 0.76),
 ('Chicago', 'Chicago', 0.8),
 ("All The King's Men", "All the King's Men", 0.7911111111111111),
 ('Across The Universe', 'Across the Universe', 0.7931537505180274),
 ('XXX', 'XX/XY', 0.7573333333333334),
 ('Quantum Of Solace', 'Quantum Heist, The', 0.6725054466230937),
 ('Max Payne', 'Max Payne', 0.76),
 ('A Walk On The Moon', 'Walk', 0.6962962962962964),
 ('Blazing Saddles', 'Blazing Saddles', 0.8),
 ('Yentl', 'Yentl', 0.8),
 ('Changeling', 'Changeling, The', 0.7733333333333334),
 ('Miss Congeniality', 'Miss Congeniality', 0.8),
 ('Casino Royale', 'Casino Royale', 0.8014814814814816),
 ('Sky High', 'Sky High', 0.8035714285714286),
 ('Madagascar: Escape 2 Africa', 'Madagascar 2', 0.7555555555555556),
 ('The Last Samurai', 'That Sun', 0.7114285714285715),
 ('Stars In My Crown', 'Stars in My Crown', 0.7905882352941177),

In [ ]:
titles_jaro_actyear = resultIMDBtitles("/content/matches_jaro.txt")
print(titles_jaro_actyear)

['Ever After', 'First Daughter', 'Hope Floats', '88 Minutes', 'Chicago', "All the King's Men", 'Across the Universe', 'XX/XY', 'Quantum Heist', 'Max Payne', 'Walk', 'Blazing Saddles', 'Yentl', 'Changeling', 'Miss Congeniality', 'Casino Royale', 'Sky High', 'Madagascar 2', 'That Sun', 'Stars in My Crown', 'Bodyguard of Lies', 'Long Kiss Goodnight', 'Hustler', 'Forever Fever', 'They Meet Again', 'Black Snake Moan', 'They Marched Into Sunlight', 'Tomorrow Never Dies', 'GoldenEye', 'Flying Guillotine', 'Raising Arizona', 'Firestorm', 'Fifth Element', 'Mamma Mia', 'Indiana Jones 4', "Hitchhiker's Guide to the Galaxy", ' Hidden Dragon', 'Batman Begins', 'Milk and Fashion', 'Mandalay', 'While You Were Sleeping', 'Portrait of Jennie', 'Sail a Crooked Ship', 'Hocus Pocus', 'Man of the House', 'Hellboy', 'Malek', 'Thieves', 'Australia', 'Stolen Moments', 'Three Bad Sisters', 'Battle Royale', 'Atemlos', 'Second Skin', 'Live Free or Die Hard', 'Trophy', 'Transporteur II', 'Exit Wounds', 'Howl', 'B

In [ ]:
result_jaro_actyear = get_imdb_ids_for_titles(titles_jaro_actyear, imdb_mapping)
print(result_jaro_actyear)

['idm425250523376', 'idm424963555104', 'idm425290058512', 'idm424707838832', 'idm425129366640', 'idm424749022656', 'idm424475490592', 'idm423818766784', 'Unknown', 'idm425628652432', 'idm423915548960', 'idm423678271392', 'idm425375491232', 'idm424396990336', 'idm425059792112', 'idm424963079504', 'idm425736378048', 'idm424940905216', 'idm425823466176', 'idm425421185088', 'idm423919240032', 'Unknown', 'idm424463031888', 'idm425252983840', 'idm425302092640', 'idm423745135568', 'idm424606490960', 'idm425871299408', 'idm425284428464', 'idm425890982240', 'idm425454828688', 'idm424772892384', 'Unknown', 'idm425298991056', 'idm423946401680', 'Unknown', 'Unknown', 'idm424936137152', 'idm424889052896', 'idm425499057008', 'idm424342441920', 'idm424936457360', 'idm425812550240', 'idm424450893328', 'idm425868692256', 'idm425299174592', 'idm424020449840', 'idm425789254336', 'idm425792646144', 'idm425294851072', 'idm425810044096', 'idm425738141616', 'idm423989040864', 'idm425647507488', 'idm424248858

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

relevant = []

with open ("ground_truth-1.txt","r") as f:
  for line in f.readlines():
    relevant.append(line.strip().split(",")[1])


resulT = pad_predictions(relevant, result_jaro_actyear)
print(resulT)
pre,rec,f1,av = precision_recall_fscore_support(relevant, resulT, average="micro")
print("Precision: ",pre,"\nRecall:",rec,"\nF1 score:",f1)

['idm425250523376', 'idm424963555104', 'idm425290058512', 'idm424707838832', '0', 'idm424749022656', 'idm424475490592', '0', '0', 'idm425628652432', '0', 'idm423678271392', 'idm425375491232', '0', 'idm425059792112', 'idm424963079504', '0', 'idm424940905216', '0', 'idm425421185088', 'idm423919240032', '0', '0', '0', '0', '0', 'idm423745135568', '0', '0', 'idm425871299408', 'idm425284428464', '0', 'idm425454828688', '0', '0', 'idm425298991056', 'idm423946401680', '0', '0', '0', '0', 'idm425499057008', 'idm424342441920', 'idm424936457360', 'idm425812550240', '0', '0', 'idm425299174592', '0', '0', '0', '0', '0', '0', '0', '0', 'idm424248858192', '0', '0', 'idm424981311680', '0', '0', 'idm425411660832', 'idm424294232976', '0', '0', '0', 'idm425132453616', '0', '0', '0', 'idm424417892160', 'idm425893080576', 'idm424137796112', '0', 'idm425875771888', '0', '0', '0', '0', 'idm425202809872', '0', 'idm424274838368', 'idm424486292848', 'idm425883246848', '0', 'idm425927693664', '0', '0', '0', '0'

8.3 Mapping

In [ ]:
import re
def standardize_title(title):
    title = title.strip()
    match = re.match(r'(.+), (The|A|An)$', title, re.IGNORECASE)
    if match:
        title = f"{match.group(2)} {match.group(1)}"

    title = re.sub(r'\s+', ' ', title).strip()
    return title

In [ ]:
def standardize_actor(actor):
    actor = actor.strip()

    if "," in actor:
        parts = [part.strip().capitalize() for part in actor.split(",")]
        actor = f"{parts[1]} {parts[0]}" if len(parts) > 1 else actor.capitalize()
    else:
        actor = " ".join(word.capitalize() for word in actor.split())

    return actor

In [ ]:
def extract_imdb_movies_with_actors(imdb_data):
    imdb_movies = {}

    imdb_datakeys = list(imdb_data.keys())
    for movie_id in imdb_datakeys:
        movie = imdb_data.get(movie_id)
        title = standardize_title(movie[0])
        year = movie[1]
        actors = {standardize_actor(actor) for actor in movie[2]}

        if movie_id:
            imdb_movies[movie_id] = (title, year, actors)

    return imdb_movies

In [ ]:
standard_names = extract_imdb_movies_with_actors(imdb_data)
print(dict(islice(standard_names.items(), 2)))

{'idm423783729296': ('Katka i Shiz', 1992, {'Georgi Drozd', 'Otar Megvinetukhutsesi', 'Armen Dzhigarkhanyan', 'Irina Metlitskaya', 'Yelena (i) Shevchenko', 'Yuri (i) Vasilyev', 'Denis Karasyov'}), 'idm423783724144': ('Prisoner of Zenda', 1988, set())}


In [ ]:
weights = {"title": 0.4, "year": 0.3, "actors": 0.3}
find_best_unique_matches_actorsyears(tv_data.values(), standard_names.values(), threshold=0.5, output_file="matches_mapping.txt", weights=weights)

Matches saved to matches_mapping.txt


[('Ever After', 'Ever After', 0.7919354838709677),
 ('First Daughter', 'First Daughter', 0.8624999999999999),
 ('Hope Floats', 'Hope Floats', 0.8166666666666667),
 ('88 Minutes', '88 Minutes', 0.7255555555555556),
 ('Chicago', 'Chicago', 0.7345132743362831),
 ("All The King's Men", "All the King's Men", 0.8013151927437642),
 ('Across The Universe', 'Across the Universe', 0.7269501264077224),
 ('XXX', 'XX/XY', 0.6573333333333333),
 ('Quantum Of Solace', 'Duality of Self', 0.5606535947712419),
 ('Max Payne', 'Max Payne', 0.67),
 ('A Walk On The Moon', 'A Walk on the Moon', 0.7972222222222222),
 ('Blazing Saddles', 'Blazing Saddles', 0.8304347826086956),
 ('Yentl', 'Yentl', 0.8676470588235293),
 ('Changeling', 'Changeling', 0.67),
 ('Miss Congeniality', 'Miss Congeniality', 0.8048192771084337),
 ('Casino Royale', 'Casino Royale', 0.812121212121212),
 ('Sky High', 'Sky High', 0.8274999999999999),
 ('Madagascar: Escape 2 Africa', 'Madagascar 2', 0.6555555555555557),
 ('The Last Samurai', 'T

In [ ]:
titles_jaro_mapping = resultIMDBtitles("/content/matches_mapping.txt")
print(titles_jaro_mapping)

['Ever After', 'First Daughter', 'Hope Floats', '88 Minutes', 'Chicago', "All the King's Men", 'Across the Universe', 'XX/XY', 'Duality of Self', 'Max Payne', 'A Walk on the Moon', 'Blazing Saddles', 'Yentl', 'Changeling', 'Miss Congeniality', 'Casino Royale', 'Sky High', 'Madagascar 2', 'The Last Samurai', 'Stars in My Crown', 'Bodyguard of Lies', 'The Long Kiss Goodnight', 'The Hustler', 'Forever Fever', 'The Great McGinty', 'Black Snake Moan', 'The Death of Harry Tobin', 'Tomorrow Never Dies', 'GoldenEye', 'Stump the Millionaire', 'Raising Arizona', 'Firestorm', 'The Fifth Element', 'Mamma Mia', 'Indiana Jones 4', "The Hitchhiker's Guide to the Galaxy", ' Hidden Dragon', 'Batman Begins', 'Milk and Fashion', 'Mandalay', 'While You Were Sleeping', 'Portrait of Jennie', 'Sail a Crooked Ship', 'Hocus Pocus', 'Man of the House', 'Hellboy', 'MacBett', 'The Devil Wears Prada', 'Australia', 'Stolen Moments', 'The Searchers', 'Eager to Die', 'ATL', 'The Boondock Saints', 'Live Free or Die Ha

In [ ]:
result_jaro_mapping = get_imdb_ids_for_titles(titles_jaro_mapping, imdb_mapping)
print(result_jaro_mapping)

['idm425250523376', 'idm424963555104', 'idm425290058512', 'idm424707838832', 'idm425129366640', 'idm424749022656', 'idm424475490592', 'idm423818766784', 'idm425345228816', 'idm425628652432', 'Unknown', 'idm423678271392', 'idm425375491232', 'idm424396990336', 'idm425059792112', 'idm424963079504', 'idm425736378048', 'idm424940905216', 'Unknown', 'idm425421185088', 'idm423919240032', 'Unknown', 'Unknown', 'idm425252983840', 'Unknown', 'idm423745135568', 'Unknown', 'idm425871299408', 'idm425284428464', 'idm424552923968', 'idm425454828688', 'idm424772892384', 'Unknown', 'idm425298991056', 'idm423946401680', 'Unknown', 'Unknown', 'idm424936137152', 'idm424889052896', 'idm425499057008', 'idm424342441920', 'idm424936457360', 'idm425812550240', 'idm424450893328', 'idm425868692256', 'idm425299174592', 'idm425335996800', 'Unknown', 'idm425792646144', 'idm425294851072', 'Unknown', 'idm423678992848', 'idm425242906288', 'Unknown', 'idm424248858192', 'idm424018044832', 'Unknown', 'idm424981311680', '

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

relevant = []

with open ("ground_truth-1.txt","r") as f:
  for line in f.readlines():
    relevant.append(line.strip().split(",")[1])


resulT = pad_predictions(relevant, result_jaro_mapping)
print(resulT)
pre,rec,f1,av = precision_recall_fscore_support(relevant, resulT, average="micro")
print("Precision: ",pre,"\nRecall:",rec,"\nF1 score:",f1)

['idm425250523376', 'idm424963555104', 'idm425290058512', 'idm424707838832', '0', 'idm424749022656', 'idm424475490592', '0', '0', 'idm425628652432', '0', 'idm423678271392', 'idm425375491232', '0', 'idm425059792112', 'idm424963079504', '0', 'idm424940905216', '0', 'idm425421185088', 'idm423919240032', '0', '0', '0', '0', '0', 'idm423745135568', '0', '0', 'idm425871299408', 'idm425284428464', 'idm424552923968', 'idm425454828688', '0', '0', 'idm425298991056', 'idm423946401680', '0', '0', '0', '0', 'idm425499057008', 'idm424342441920', 'idm424936457360', 'idm425812550240', '0', '0', 'idm425299174592', '0', '0', '0', '0', '0', '0', 'idm425242906288', '0', 'idm424248858192', 'idm424018044832', '0', 'idm424981311680', '0', '0', 'idm425411660832', 'idm424294232976', '0', '0', '0', 'idm425132453616', '0', '0', '0', 'idm424417892160', 'idm425893080576', 'idm424137796112', '0', 'idm425875771888', '0', '0', '0', '0', 'idm425202809872', '0', 'idm424274838368', 'idm424486292848', 'idm425883246848', 

8.4) Merging

a, b, c together

In [ ]:
import xml.sax
from xml.sax.handler import ContentHandler
import json
import ast
import re

class IMDBHandlerCompleteALL(ContentHandler):
    def __init__(self):
        super().__init__()
        self.current_tag = None
        self.current_id = None
        self.current_title = None
        self.current_year = None
        self.current_genre = []
        self.current_actors = []
        self.current_roles = []
        self.current_director = None
        self.current_location = None
        self.current_keywords = []
        self.current_plot = None
        self.movie_dict = {}
        self.content_buffer = ""
        self.in_movie = False
        self.in_actor = False

    def startElement(self, name, attrs):
        self.current_tag = name
        if name == "movie":
            self.in_movie = True
            if attrs.get("imdbid"):
                self.current_id = attrs.get("imdbid")
            self.current_title = None
            self.current_year = None
            self.current_genre = []
            self.current_actors = []
            self.current_roles = []
            self.current_director = None
            self.current_location = None
            self.current_keywords = []
            self.current_plot = None
        elif name == "actor":
            self.in_actor = True

        self.content_buffer = ""

    def endElement(self, name):
        if self.in_movie:
            if name == "title":
                self.current_title = self.content_buffer.strip()
            elif name == "year":
                try:
                    self.current_year = int(self.content_buffer.strip())
                except (ValueError, TypeError):
                    self.current_year = None
            elif name == "genre":
                if self.content_buffer.strip():
                    self.current_genre.append(self.content_buffer.strip())
            elif name == "director":
                self.current_director = self.content_buffer.strip()
            elif name == "location":
                self.current_location = self.content_buffer.strip()
            elif name == "keyword":
                if self.content_buffer.strip():
                    self.current_keywords.append(self.content_buffer.strip())
            elif name == "plot":
                self.current_plot = self.content_buffer.strip()
            elif name == "name" and self.in_actor:
                if self.content_buffer.strip():
                    self.current_actors.append(self.content_buffer.strip())
            elif name == "role" and self.in_actor:
                if self.content_buffer.strip():
                    self.current_roles.append(self.content_buffer.strip())
            elif name == "actor":
                self.in_actor = False

            elif name == "movie":
                if self.current_id and self.current_title:
                    movie_data = {
                        'title': self.current_title,
                        'year': self.current_year,
                        'imdbid': self.current_id,
                        'genres': self.current_genre,
                        'director': self.current_director,
                        'location': self.current_location,
                        'keywords': self.current_keywords,
                        'plot': self.current_plot,
                        'actors': []
                    }

                    for i in range(min(len(self.current_actors), len(self.current_roles))):
                        movie_data['actors'].append({
                            'name': self.current_actors[i],
                            'role': self.current_roles[i]
                        })

                    self.movie_dict[self.current_id] = movie_data
                self.in_movie = False

        self.content_buffer = ""
        if name == self.current_tag:
            self.current_tag = None

    def characters(self, content):
        if self.current_tag:
            self.content_buffer += content

    def get_movie_dict(self):
        return self.movie_dict

In [ ]:
from xml.dom import minidom

def get_text(element):
    return element.firstChild.nodeValue.strip() if element and element.firstChild and element.firstChild.nodeValue else None

# here we parse all of the attributes and saving them in a dictionary
def parse_tv_guide_ALL(file):
    try:
        doc = minidom.parse(file)
        movie_elements = doc.getElementsByTagName("movie")

        movie_dict = {}  # {tv_id: movie_data}

        for movie in movie_elements:
            tv_id = movie.getAttribute("tvgid")
            if not tv_id:
                continue

            title = get_text(movie.getElementsByTagName("name")[0]) if movie.getElementsByTagName("name") else None
            year = get_text(movie.getElementsByTagName("year")[0]) if movie.getElementsByTagName("year") else None
            rating = get_text(movie.getElementsByTagName("rating")[0]) if movie.getElementsByTagName("rating") else None
            country = get_text(movie.getElementsByTagName("country")[0]) if movie.getElementsByTagName("country") else None
            running_time = get_text(movie.getElementsByTagName("running-time")[0]) if movie.getElementsByTagName("running-time") else None
            format_type = get_text(movie.getElementsByTagName("format")[0]) if movie.getElementsByTagName("format") else None
            review_rating = get_text(movie.getElementsByTagName("review-rating")[0]) if movie.getElementsByTagName("review-rating") else None

            genres = set()
            for genre in movie.getElementsByTagName("genre"):
                genre_text = get_text(genre)
                if genre_text:
                    genres.add(genre_text)

            production_companies = set()
            for prod in movie.getElementsByTagName("production-company"):
                prod_text = get_text(prod)
                if prod_text:
                    production_companies.add(prod_text)

            released_by = get_text(movie.getElementsByTagName("released-by")[0]) if movie.getElementsByTagName("released-by") else None

            user_rating = get_text(movie.getElementsByTagName("rating")[0]) if movie.getElementsByTagName("rating") else None
            rating_support = get_text(movie.getElementsByTagName("support")[0]) if movie.getElementsByTagName("support") else None

            actors = []
            for actor in movie.getElementsByTagName("actor"):
                actor_name = get_text(actor.getElementsByTagName("name")[0]) if actor.getElementsByTagName("name") else None
                actor_role = get_text(actor.getElementsByTagName("role")[0]) if actor.getElementsByTagName("role") else None
                if actor_name and actor_role:
                    actors.append({"name": actor_name, "role": actor_role})

            credits = []
            for credit in movie.getElementsByTagName("credit"):
                credit_name = get_text(credit.getElementsByTagName("name")[0]) if credit.getElementsByTagName("name") else None
                credit_role = get_text(credit.getElementsByTagName("role")[0]) if credit.getElementsByTagName("role") else None
                if credit_name and credit_role:
                    credits.append({"name": credit_name, "role": credit_role})

            review_paragraphs = []
            for paragraph in movie.getElementsByTagName("paragraph"):
                paragraph_text = get_text(paragraph)
                if paragraph_text:
                    review_paragraphs.append(paragraph_text)

            movie_data = {
                "title": title,
                "year": int(year) if year and year.isdigit() else None,
                "rating": rating,
                "country": country,
                "running_time": int(running_time) if running_time and running_time.isdigit() else None,
                "format": format_type,
                "genres": list(genres),
                "production_companies": list(production_companies),
                "released_by": released_by,
                "user_rating": float(user_rating) if user_rating and user_rating.replace(".", "").isdigit() else None,
                "rating_support": int(rating_support) if rating_support and rating_support.isdigit() else None,
                "actors": actors,
                "credits": credits,
                "review": " ".join(review_paragraphs)
            }

            # store in main dictionary
            movie_dict[tv_id] = movie_data

        return movie_dict

    except Exception as e:
        print(f"Error in parse_tv_guide_ALL: {e}")
        return {}  # Return empty dictionary on error


In [ ]:
def parse_imdb_complete(file_path):
    handler = IMDBHandlerCompleteALL()
    xml.sax.parse(file_path, handler)
    return handler.get_movie_dict()

In [ ]:
def normalize_name(name):
    if isinstance(name, list): # Added this condition to handle the list of actors
        return [normalize_name(actor['name']) for actor in name]
    actor = name.strip()

    if "," in actor:
        parts = [part.strip().capitalize() for part in actor.split(",")]
        actor = f"{parts[1]} {parts[0]}" if len(parts) > 1 else actor.capitalize()
    else:
        actor = " ".join(word.capitalize() for word in actor.split())

    return actor

In [ ]:
def merge_lists(list1, list2, key):
    merged_dict = {d[key]: d for d in list1 if key in d}  # store first list in a dictionary

    for item in list2:
        if key in item:
            merged_dict[item[key]] = item  # overwrite with second list's values

    return list(merged_dict.values())

In [ ]:
def merge_movies(tv_guide_movies, imdb_movies, matches_file_path):
    merged_movies = {}

    with open(matches_file_path, 'r') as file:
        matches = file.read()

    match_dict = {}
    for line in matches.strip().split('\n'):
        parts = line.split(',', 2)
        if len(parts) >= 2:
            tv_title = parts[0].strip()
            imdb_title = parts[1].strip()

            # get the score
            score = None
            if len(parts) > 2:
                try:
                    score = float(parts[2])
                except ValueError:
                    pass

            match_dict[tv_title] = (imdb_title, score)

    # Create a title-to-ID mapping for IMDB movies
    title_to_id = {movie_data['title']: imdb_id for imdb_id, movie_data in imdb_movies.items()}


    # going through each TV guide movie
    for tv_id, tv_movie in tv_guide_movies.items():
        if not isinstance(tv_movie, dict):
            print(f"Skipping invalid movie entry: {tv_title}")
            continue

        tv_title = tv_guide_movies[tv_id]['title']
        # find matching IMDB movie
        imdb_id = None
        if tv_title in match_dict.keys():
            imdb_title, _ = match_dict[tv_title]
            imdb_id = title_to_id.get(imdb_title)

        if imdb_id and imdb_id in imdb_movies:
            imdb_movie = imdb_movies[imdb_id]

            # A: Merge single-valued attributes from IMDB (overwrite TV Guide values if different)
            tv_movie['title'] = imdb_movie.get('title', tv_title)  # Overwrite title if different
            tv_movie['year'] = str(imdb_movie.get('year', tv_movie.get('year', '')))

            for key in ['director', 'location', 'keywords', 'plot']:
                if imdb_movie.get(key):
                    tv_movie[key + 's'] = imdb_movie[key]

            # add IMDB ID
            tv_movie['imdbid'] = imdb_id

            # B: Merge genres (union of genres from both sources)
            tv_genres = tv_movie.get('genres', [])
            imdb_genres = imdb_movie.get('genres', [])

            if isinstance(tv_genres, str):
                tv_genres = [tv_genres]

            all_genres = set(tv_genres) | set(imdb_genres)
            tv_movie['genres'] = list(all_genres)

            # C: Merge actors
            tv_actors = tv_movie.get('actors', {})
            imdb_actors = imdb_movie.get('actors', [])


            if isinstance(tv_actors, dict):  # Convert single actor to list
                tv_actors = [tv_actors]

            for actor in imdb_actors:
                if isinstance(actor, dict) and 'name' in actor:
                  actor['name'] = normalize_name(actor['name'])


            # dictionary to store normalized actors (to avoid duplicates)
            normalized_actors = {normalize_name(actor['name']): actor for actor in tv_actors if isinstance(actor, dict) and 'name' in actor}


            merged_list = merge_lists(tv_actors, normalized_actors, "role")
            print(merged_list)

            # update actor list in TV movie
            tv_movie['actors'] = merged_list

        # store the merged movie
        merged_movies[tv_title] = tv_movie

    return merged_movies


In [ ]:
# just to inspect if it works, but uneccessary => inspect JSON file is better
def inspect_sample(merged_movies, sample_size=3):
    print(f"\nSample of {sample_size} merged movies for inspection:")

    movies_list = list(merged_movies.values())[:sample_size]

    for i, movie in enumerate(movies_list):
        print(f"\n--- Movie {i+1} ---")
        print(f"Title: {movie.get('name')}")
        print(f"Year: {movie.get('year')}")
        print(f"IMDB ID: {movie.get('imdbid', 'Not available')}")

        if 'genres' in movie and 'genre' in movie['genres']:
            genres = movie['genres']['genre']
            if isinstance(genres, list):
                print(f"Genres: {', '.join(genres)}")
            else:
                print(f"Genres: {genres}")

        if 'directors' in movie:
            print(f"Directors: {movie['directors']}")

        if 'locations' in movie:
            print(f"Locations: {movie['locations']}")

        if 'actor-list' in movie and 'actor' in movie['actor-list']:
            actors = movie['actor-list']['actor']
            if isinstance(actors, list):
                print(f"Sample actors ({min(3, len(actors))} of {len(actors)}):")
                for actor in actors[:3]:
                    if isinstance(actor, dict):
                        print(f"  - {actor.get('name', 'Unknown')}: {actor.get('role', 'Unknown role')}")
            else:
                if isinstance(actors, dict):
                    print(f"Actor: {actors.get('name', 'Unknown')}: {actors.get('role', 'Unknown role')}")


In [ ]:
# File paths
tv_guide_file = "tvguide-ds.xml"
imdb_file = "imdb-ds.xml"
matches_file = "/content/matches_mapping.txt"


print("Parsing TV Guide data...")
try:
    tv_guide_movies = parse_tv_guide_ALL(tv_guide_file)
    print(f"Successfully parsed {len(tv_guide_movies)} TV Guide movies")
except Exception as e:
    print(f"Error parsing TV Guide data: {e}")
    # for testing, create a sample TV Guide movie
    tv_guide_movies = [ast.literal_eval("""{'name': '88 Minutes',
 'rank': '86',
 'year': '2008',
 'rating': 'R',
 'country': 'U.S.',
 'running-time': '108',
 'format': 'Color',
 'genres': {'genre': 'Thriller'},
 'production-companies': {'production-company': ['Equity Pictures Medienfonds',
   'Millennium Films',
   'Nu Image Entertainment']},
 'released-by': 'Sony',
 'user-rating': {'rating': '3.5', 'support': '10'},
 'actor-list': {'actor': [{'name': 'Carrie Genzel',
    'role': 'Stephanie Parkman'},
   {'name': 'Christopher Redman', 'role': 'Jeremy Guber'},
   {'name': 'Al Pacino', 'role': 'Jack Gramm'}]}}"""
    )]

Parsing TV Guide data...
Successfully parsed 98 TV Guide movies


In [ ]:
# Load IMDB data
print("Parsing IMDB data...")
try:
    imdb_movies = parse_imdb_complete(imdb_file)
    print(f"Successfully parsed {len(imdb_movies)} IMDB movies")
except Exception as e:
    print(f"Error parsing IMDB data: {e}")
    # for testing, create a sample IMDB movie
    imdb_movies = {
        'idm123456': {
            'title': '88 Minutes',
            'year': 2008,
            'imdbid': 'idm123456',
            'genres': ['Thriller', 'Crime'],
            'director': 'Jon Avnet',
            'location': 'Seattle, Washington, USA',
            'keywords': ['forensic psychiatrist', 'death row', 'countdown'],
            'plot': 'On the day a serial killer is set to be executed, he claims he is innocent and a forensic psychiatrist has 88 minutes to find the real killer.',
            'actors': [
                {'name': 'Al Pacino', 'role': 'Jack Gramm'},
                {'name': 'Alicia Witt', 'role': 'Kim Cummings'},
                {'name': 'Neal McDonough', 'role': 'Jon Forster'}
            ]
        }
    }

Parsing IMDB data...
Successfully parsed 243856 IMDB movies


In [ ]:
# merge movies
print("Merging movies based on matches...")
merged_movies = merge_movies(tv_guide_movies, imdb_movies, matches_file)
print(f"Successfully merged {len(merged_movies)} movies")

# inspect_sample(merged_movies)

# Save the merged data to a JSON file
with open("merged_movies.json", "w") as f:
    json.dump(merged_movies, f, indent=2)
print("\nMerged data saved to 'merged_movies.json'")

Merging movies based on matches...
Ever After
[{'name': 'Matyelok Gibbs', 'role': 'Louise'}, {'name': 'Joerg Stadler', 'role': 'Wilhelm Grimm'}, {'name': 'Elvira Stevenson', 'role': 'Queen of Spain'}, {'name': 'Toby Jones', 'role': 'Royal Page'}, {'name': 'Megan Dodds', 'role': 'Marguerite'}, {'name': 'Lee Ingley', 'role': 'Gustave'}, {'name': 'Anna Maguire', 'role': 'Young Danielle'}, {'name': 'Kate Lansbury', 'role': 'Paulette'}, {'name': 'Mark Lewis', 'role': 'Gypsy Leader'}, {'name': 'Timothy West', 'role': 'King Francis'}, {'name': 'Francois Velter', 'role': 'Choirman'}, {'name': 'Susan Field', 'role': 'Laundry Supervisor'}, {'name': 'Ricki Cuttell', 'role': 'Young Gustave'}, {'name': 'Alex Pooley', 'role': 'Young Jacqueline'}, {'name': 'Erick Awanzino', 'role': 'Short Bald Man'}, {'name': 'Elizabeth Earl', 'role': 'Young Marguerite'}, {'name': 'Rupam Maxwell', 'role': 'Marquis de Limonges'}, {'name': 'Judy Parfitt', 'role': 'Queen Marie'}, {'name': 'Walter Sparrow', 'role': 'Maur